In [1]:
import numpy as np
import mvt_estimation
from scipy.stats import chi2

In [6]:
def generate_mvt(n, nu, mu, Sigma):
    p = mu.shape[0]

    # Generate multivariate normal
    N = np.random.randn(n, p)

    # Generate Chi-squared
    rng = np.random.default_rng()
    Chi = rng.chisquare(df=nu, size=n)

    # Combine to get multivariate T
    T_tilde = np.sqrt(nu) * N / np.sqrt(Chi)[:, None]

    # Apply transformation
    A = np.linalg.cholesky(Sigma)
    T = mu + T_tilde @ A.T

    return T

In [10]:
def reject_h0_nu(theta, omega, n, critical_value):
    # Create A (1x15) matrix full of zeros and only 1 at the last position.
    # 1 is on the last position because we are testing nu, the parameter that
    # is the last in our Theta
    A = np.zeros(15).T
    A[14] = 1

    # Create b, it's (1x1) because we have only one estimator to check
    b = 4

    # Difference (A @ theta - b) has size (1x1)
    diff = A @ theta - b

    # A @ omega @ A.T has also size (1) as well as its inverse
    AOA = A @ omega @ A.T
    AOA_inv = 1 / AOA

    # Calculate the test statistic
    statistic = n * diff.T * AOA_inv * diff

    # We compare test statistic with critical value of Chi-squared
    # with 1 degree of freedom and alpha=0.05 (9.4877).
    # We reject H0 if statistic is higher than critical value.
    return statistic > critical_value

In [11]:
def extract_estimated_results(p, theta_hat):
    # First p elements is mu
    mu = theta_hat[0:p]

    # Last element is nu
    nu = theta_hat[-1]

    # Everything in between is lower triangle of Sigma
    a = theta_hat[p:-1].flatten()
    sigma = np.zeros((p, p))
    idx = np.tril_indices(p)
    sigma[idx] = a

    # Combine lower triangle of Sigma to get full Sigma
    sigma = sigma + sigma.T - np.diag(np.diag(sigma))

    return mu, sigma, nu

In [25]:
def calculations_for_n(n,p, nu, mu, Sigma, critical_value):
    print(f"\n### Start for n = {n}")
    reject = 0

    mu_sum = 0
    sigma_sum = np.zeros((p, p))
    nu_sum = 0

    mu1_list = []
    se_mu1_list = []

    sigma32_list = []
    se_sigma32_list = []

    nu_list = []
    se_nu_list = []

    m = 1000
    for i in range(0, m):
        # Generate MVT with given parameters
        T = generate_mvt(n, nu, mu, Sigma)

        # Estimate parameters of MVT
        theta, VCV_sam, VCV_asy, _ = mvt_estimation.MVT_MLE_approach2(T)
        theta = np.asarray(theta).ravel()
        mu_hat, Sigma_hat, nu_hat = extract_estimated_results(p, theta)

        # Update sum of estimated params for calculating average
        mu_sum += mu_hat
        sigma_sum += Sigma_hat
        nu_sum += nu_hat

        # Update lists with estimated parameters and their variance
        se = np.sqrt(np.diag(VCV_sam))
        mu1_list.append(mu_hat[0])
        se_mu1_list.append(se[0])
        sigma32_list.append(Sigma_hat[2][1])
        se_sigma32_list.append(se[8])
        nu_list.append(nu_hat)
        se_nu_list.append(se[-1])

        # Test H0: nu=4
        if reject_h0_nu(theta, VCV_asy, n, critical_value):
            reject += 1

    # Calculate rejection rate for the H0
    rejection_rate = reject / m
    print(f"For {m} iteration for sample size of {n} the rejection rate is {rejection_rate}\n")

    # Calculate average per estimated parameter across all m runs
    mu_average = mu_sum / m
    sigma_average = sigma_sum / m
    nu_average = nu_sum / m

    print(f"Mu average: \n{mu_average}\n")
    print(f"Sigma average: \n{sigma_average}\n")
    print(f"Nu average: {nu_average}\n")

    # Calculate bias per estimated parameter
    mu1_bias = mu_average[0] - mu[0]
    sigma32_bias = sigma_average[2][1]-Sigma[2][1]
    nu_bias = nu_average - nu

    print(f"Mu_1 bias: {mu1_bias}\n")
    print(f"Sigma32 bias: {sigma32_bias}\n")
    print(f"Nu bias: {nu_bias}\n")

    emp_sd_mu1 = np.std(mu1_list, ddof=1)
    avg_se_mu1 = np.mean(se_mu1_list)
    emp_sd_sigma32 = np.std(sigma32_list, ddof=1)
    avg_se_sigma32 = np.mean(se_sigma32_list)
    emp_sd_nu = np.std(nu_list, ddof=1)
    avg_se_nu = np.mean(se_nu_list)

    print(f"Empirical SD mu: {emp_sd_mu1:.2f} Average SE: {avg_se_mu1:.2f} Ratio: {(avg_se_mu1 / emp_sd_mu1):.2f}")

    print(f"Empirical SD Sigma 32: {emp_sd_sigma32:.2f} Average SE: {avg_se_sigma32:.2f} Ratio: {(avg_se_sigma32 / emp_sd_sigma32):.2f}")

    print(f"Empirical SD nu: {emp_sd_nu:.2f} Average SE: {avg_se_nu:.2f} Ratio: {(avg_se_nu / emp_sd_nu):.2f}")

## Run the simulations for given set of parameters and different sample sizes

In [28]:
# Parameters given in the assigment
n_options = [50, 100, 500, 750]
nu = 4
mu = np.array([1, 2, -1, 3])
Sigma = np.array([[2, 0, 1, 1],
                  [0, 3, 2, 1],
                  [1, 2, 5, 2],
                  [1, 1, 2, 6]])

# We use rank of A matrix to find degrees of freedom for Chi squared
# distribution. In our case df=1 as we are testing only one condition
df_chi = 1
alpha = 0.05
critical_value = chi2.ppf(1 - alpha, df=1)

for n in n_options:
    calculations_for_n(n=n, p=4, nu=nu, mu=mu, Sigma=Sigma, critical_value=critical_value)

### Start for n = 50
For 1000 iteration for sample size of 50 the rejection rate is 0.046

Mu average: 
[ 1.01019676  1.98525282 -1.00958367  2.990373  ]

Sigma average: 
[[ 2.06075172 -0.0325131   1.00009684  1.01195259]
 [-0.0325131   3.11660739  2.04276629  1.01657769]
 [ 1.00009684  2.04276629  5.11762005  2.03675984]
 [ 1.01195259  1.01657769  2.03675984  6.18018178]]

Nu average: 24.417917449629066

Mu_1 bias: 0.010196758124091376

Sigma32 bias: 0.04276629227665385

Nu bias: 20.417917449629066

Empirical SD mu: 0.22 Average SE: 0.23 Ratio: 1.01
Empirical SD Sigma 32: 0.84 Average SE: 0.76 Ratio: 0.91
Empirical SD nu: 316.45 Average SE: 503.65 Ratio: 1.59
### Start for n = 100
For 1000 iteration for sample size of 100 the rejection rate is 0.051

Mu average: 
[ 0.99619843  2.00121066 -1.01016946  2.99599686]

Sigma average: 
[[ 2.03123431 -0.00780626  1.01337829  1.01574664]
 [-0.00780626  3.04426391  2.02565141  1.01108914]
 [ 1.01337829  2.02565141  5.08709115  2.01963171]
 [ 1.